**Forward:**

$$
\frac{\partial u}{\partial t} = D \frac{\partial^2 u}{\partial x^2}, \quad x \in [-1, 1], \quad t \in [0, 1]
$$

Boundary conditions (Dirichlet): $$u(-1, t) = u(1, t) = 0$$

Initial conditions: $$u(x, 0) = \sin(\pi x)$$

**Backwardа:**

It is assumed that the true value of \(D\) is unknown. We generate synthetic noisy “experimental” data based on the analytical solution with \(D_{true} = 1.0\). The goal of PINN is to find the value of \(D_{pred}\) by minimizing the residual of the equation and the error relative to the observations.

In [19]:
import numpy as np
import torch
import deepxde as dde

from matplotlib import pyplot as plt

D = dde.Variable(1.0)


### Define the initial conditions of our problem and create the space in which the solution will take place (the combination of geometric and temporal spaces).

In [20]:
geometric_interval = dde.geometry.Interval(-1, 1)
time_interval = dde.geometry.TimeDomain(0, 1)

full_domain = dde.geometry.GeometryXTime(geometric_interval, time_interval)

### PDE function and the residual of the resulting equation.

In [21]:
def pde (x, y):
    dy_t = dde.grad.jacobian(y, x, i=0, j=1)
    dy_xx = dde.grad.hessian(y, x, i=0, j=0)

    res = dy_t - D * dy_xx

    return res

### Specifying the boundary and initial conditions of the equation.

In [22]:
boundary_condition = dde.icbc.DirichletBC(
    full_domain, 
    func=lambda x: 0,
    on_boundary=lambda x, on_boundary: on_boundary
)

initial_conditions = dde.icbc.IC(
    full_domain,
    func=lambda x: np.sin(np.pi * x[:, 0:1]),
    on_initial=lambda x, on_initial: on_initial
)

### Adding “experimental data” and assembling the Data object (creating PINN training Config)

In [24]:
def analytical_solution (x):
    return np.sin(np.pi * x[:, 0:1]) * np.exp(- (np.pi**2) * x[:, 1:2])

observe_x = np.random.uniform(-1, 1, (20, 1))
observe_t = np.random.uniform(0, 1, (20, 1))
observe_domain = np.hstack((observe_x, observe_t))

observe_u = analytical_solution(observe_domain) + 0.01 * np.random.randn(20, 1)

data = dde.data.TimePDE(
    full_domain,
    pde,
    [boundary_condition, initial_conditions],
    num_domain=2500,
    num_boundary=100,
    num_initial=250,
    anchors=observe_domain
)

### Model train

In [25]:
net = dde.nn.FNN([2] + [40] * 2 + [1], "tanh", "Glorot uniform")

model = dde.Model(data, net)

model.compile("adam", lr=0.001, external_trainable_variables=[D])

variable_callback = dde.callbacks.VariableValue(
    D, 
    period=500, 
    filename="variables.dat"
)

history = model.train(iterations=2000, callbacks=[variable_callback])

model.compile("L-BFGS", external_trainable_variables=[D])
history_improvment = model.train(iterations=5000, callbacks=[variable_callback])

Compiling model...
'compile' took 0.000152 s

Training model...

Step      Train loss                        Test loss                         Test metric
0         [7.79e-03, 8.37e-03, 5.70e-01]    [7.79e-03, 8.37e-03, 5.70e-01]    []  
1000      [1.37e-03, 9.27e-04, 1.12e-03]    [1.37e-03, 9.27e-04, 1.12e-03]    []  
2000      [4.35e-04, 5.97e-05, 7.82e-05]    [4.35e-04, 5.97e-05, 7.82e-05]    []  

Best model at step 2000:
  train loss: 5.73e-04
  test loss: 5.73e-04
  test metric: []

'train' took 27.335874 s

Compiling model...
'compile' took 0.000112 s

Training model...

Step      Train loss                        Test loss                         Test metric
2000      [4.35e-04, 5.97e-05, 7.82e-05]    [4.35e-04, 5.97e-05, 7.82e-05]    []  
2315      [8.17e-07, 1.89e-07, 8.00e-07]    [8.17e-07, 1.89e-07, 8.00e-07]    []  

Best model at step 2315:
  train loss: 1.81e-06
  test loss: 1.81e-06
  test metric: []

'train' took 28.262455 s

